In [42]:
import yaml
import sys, os
sys.path.insert(0, os.path.abspath("/cs/usr/roeizucker/new_storage/jupyter_notebooks/Tom_Hope_Project/refactored_code"))
# from src.data_extractor import extract_data
from datasets import Dataset, DatasetDict, load_from_disk
import pandas as pd
from src.data_extractor import create_grouped_methyl_dfs, load_dfs, combine_cpg_dfs


In [28]:


# from utils.dataset_utils import create_dataset, create_dataset_dict, create_tokenizer
cfg_daat_extraction = yaml.safe_load(open("../configs/config_extraction_full_window_filtration.yaml"))
cfg_filtration =  yaml.safe_load(open("../configs/config_extraction_intermidiate_unique_regions_split.yaml"))

In [5]:
train_path = cfg_daat_extraction["paths"]["hf_dataset_train_path"]
ds_train = load_from_disk(train_path)
ds_test = load_from_disk(cfg_daat_extraction["paths"]["hf_dataset_test_path"])

In [23]:
df_train = pd.read_csv(cfg_filtration["paths"]["train_path"])
df_test = pd.read_csv(cfg_filtration["paths"]["test_path"])

In [29]:
paths = cfg_filtration["paths"]
task = cfg_filtration["task"]
raw_files_paths = paths["raw_data_paths"]
chroms = task.get("chromosomes", None)
full_position_column_name = "full_position"
test_size = task.get("test_size", 0.2)
random_state = cfg_filtration.get("random_state", 42)
train_path  = cfg_filtration["paths"]["train_path"]
test_path  = cfg_filtration["paths"]["test_path"]
grouped_df = create_grouped_methyl_dfs(raw_files_paths,chroms,full_position_column_name)

Processing chromosome: chr1 with length: 248956422
Processing chromosome: chr2 with length: 242193529
Processing chromosome: chr3 with length: 198295559
Processing chromosome: chr4 with length: 190214555
Processing chromosome: chr5 with length: 181538259
Processing chromosome: chr1 with length: 248956422
Processing chromosome: chr2 with length: 242193529
Processing chromosome: chr3 with length: 198295559
Processing chromosome: chr4 with length: 190214555
Processing chromosome: chr5 with length: 181538259
Processing chromosome: chr1 with length: 248956422
Processing chromosome: chr2 with length: 242193529
Processing chromosome: chr3 with length: 198295559
Processing chromosome: chr4 with length: 190214555
Processing chromosome: chr5 with length: 181538259
max_diff_quantile_value 10.0


In [31]:
grouped_df.to_csv("temp_grouped_df.csv")

In [33]:
high_diff_quantile_value = grouped_df["high_diff"].quantile(0.95)
print(high_diff_quantile_value)

0.08833922261484099


In [36]:
curr_df = grouped_df.reset_index()
high_diff_df = curr_df[curr_df["high_diff"] > high_diff_quantile_value]
low_diff_df = curr_df[curr_df["high_diff"] <= high_diff_quantile_value]

In [37]:
print("high_diff_len",len(high_diff_df))
print("low_diff_len",len(low_diff_df))

high_diff_len 9580
low_diff_len 182028


In [39]:
dfs = load_dfs(chroms=chroms,raw_files_paths=raw_files_paths,full_position_column_name=full_position_column_name)

Processing chromosome: chr1 with length: 248956422
Processing chromosome: chr2 with length: 242193529
Processing chromosome: chr3 with length: 198295559
Processing chromosome: chr4 with length: 190214555
Processing chromosome: chr5 with length: 181538259
Processing chromosome: chr1 with length: 248956422
Processing chromosome: chr2 with length: 242193529
Processing chromosome: chr3 with length: 198295559
Processing chromosome: chr4 with length: 190214555
Processing chromosome: chr5 with length: 181538259
Processing chromosome: chr1 with length: 248956422
Processing chromosome: chr2 with length: 242193529
Processing chromosome: chr3 with length: 198295559
Processing chromosome: chr4 with length: 190214555
Processing chromosome: chr5 with length: 181538259


In [43]:
labels = []  # same length/order as `kept`
for i in range(len(dfs)):
    labels.append(f"ind_{i}")

df = combine_cpg_dfs(full_position_column_name, dfs, labels)

In [48]:
count = 0
dfs_num = len(dfs)
for i in range(dfs_num):
    for j in range(i,dfs_num):
        if i == j:
            continue
        count+=1
        df[f"{i}-{j}"] = (df[f"methyl_rate_ind_{i}"] - df[f"methyl_rate_ind_{j}"]).abs()
df["max_diff"] = df[df.columns[-count:]].max(axis=1)

In [52]:
df[["max_diff"]].describe().apply(lambda s: s.apply('{0:.5f}'.format))

,max_diff
count,197833108.00000
mean,2.02076
std,6.07144
min,0.00000
25%,0.00000
50%,0.00000
75%,0.00000
max,100.00000


In [58]:
df[["methyl_rate_ind_0","methyl_rate_ind_1","methyl_rate_ind_2"]].std()

methyl_rate_ind_0    17.119793
methyl_rate_ind_1    16.758154
methyl_rate_ind_2    17.146380
dtype: float64

In [60]:
df.to_csv("/sci/archive/michall/roeizucker/temp_result_df.csv")